In [6]:
import yfinance as yf
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from src.utils import convert_dates
from src.database import SessionLocal
from src.models.committee import Committee
from src.models.trade import Trade
from src.models.legislator import Legislator


with SessionLocal() as session:
    committees = session.query(Committee).all()
    trades = session.query(Trade).all()

tdf = pd.DataFrame([trade.__dict__ for trade in trades])
tdf = tdf[['security_ticker', 'id', 'legislator_id', 'trade_type', 'disclosure_date', 'trade_date']]
tdf = tdf.dropna(subset=['security_ticker'])

In [2]:

def fetch_ticker_history(tickers, period='3mo'):
    """
    Fetches historical data for a list of tickers from yfinance and formats it into a clean DataFrame
    
    Parameters:
    ticker_series (pd.Series): Series containing tickers (can contain None values)
    period (str): Time period to fetch ('1d','5d','1mo','3mo','6mo','1y','2y','5y','10y','ytd','max')
    
    Returns:
    pd.DataFrame: Clean DataFrame with Date index and ticker data
    """
    
   
    # Download data
    df = yf.download(tickers, group_by='Ticker', period=period)
    
    # Stack and reset index
    df = df.stack(level=0).rename_axis(['Date', 'Ticker']).reset_index(level=1)
    
    # df columns to lower case
    df.columns = df.columns.str.lower()

    # index name to lower case
    df.index.name = df.index.name.lower()

    return df

In [14]:
df = fetch_ticker_history(tdf['security_ticker'].drop_duplicates().tolist(), period='6mo')

[*********             19%                       ]  36 of 186 completedFailed to get ticker 'BRK/B' reason: Expecting value: line 1 column 1 (char 0)
[*********************100%***********************]  186 of 186 completed

12 Failed downloads:
['PDRDY', 'CEQP', 'CS', 'AUY', 'DWAC', 'LSXMA']: YFPricesMissingError('$%ticker%: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")')
['BRK/B']: JSONDecodeError('Expecting value: line 1 column 1 (char 0)')
['EUN', 'ITIP']: YFPricesMissingError('$%ticker%: possibly delisted; no price data found  (period=6mo)')
['SNRE', 'LBYAV', 'EQV']: YFInvalidPeriodError("%ticker%: Period '6mo' is invalid, must be one of ['1d', '5d', '1mo', '3mo', 'ytd', 'max']")
/tmp/ipykernel_52898/1222980494.py:18: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adop

In [15]:
df

Price,ticker,open,high,low,close,volume,adj close
date,,,,,,,
2024-06-26,AAPL,211.023308,214.375735,210.165245,212.769363,66213200.0,NaN
2024-06-26,AAXJ,70.899821,70.958960,70.682971,70.840683,258700.0,NaN
2024-06-26,ABBV,167.403099,169.428189,167.265473,168.248520,5576100.0,NaN
2024-06-26,ABT,104.045046,104.599483,103.282706,103.837135,5406600.0,NaN
2024-06-26,ACN,297.399905,303.170979,296.914001,302.060394,3162100.0,NaN
...,...,...,...,...,...,...,...
2024-12-26,WFC,71.430000,71.638496,71.110001,71.279999,2937214.0,NaN
2024-12-26,XBI,91.309998,92.709999,91.027100,92.559998,3696526.0,NaN
2024-12-26,XONE,49.650002,49.654999,49.632702,49.645000,38947.0,NaN


In [16]:
tdf

,security_ticker,id,legislator_id,trade_type,disclosure_date,trade_date
14,VFC,59,42,TradeType.BUY,2024-11-12,2024-10-24
15,VWO,60,43,TradeType.SELL,2024-12-18,2024-11-20
16,SWKS,61,43,TradeType.SELL,2024-12-18,2024-11-20
17,BBEU,62,43,TradeType.SELL,2024-12-18,2024-11-20
18,SPY,63,43,TradeType.BUY,2024-12-17,2024-11-13
...,...,...,...,...,...,...
595,GBDC,640,102,TradeType.SELL,2024-08-07,2023-06-30
596,IWF,641,102,TradeType.SELL,2024-08-07,2023-06-19
597,IWF,642,102,TradeType.SELL,2024-08-07,2023-06-19
598,XBI,643,102,TradeType.SELL,2024-08-07,2023-06-30


In [ ]:
with SessionLocal() as session:
    result = session.query(Trade, Legislator).join(Legislator).all()